# FormCoach · Model A: data + training (Colab, GPU)
**Runtime → Change runtime type → T4 GPU.** Everything is saved to Google Drive in `MyDrive/formcoach/`, so you can stop and come back.

Data used:
1. **Our own recordings** (required; the only source of the test set) → upload clips to `MyDrive/formcoach/raw/` named `p03_squat_knee_valgus_t1.mp4`.
2. **Zenodo squat images** (CC BY 4.0, Good / Bad Back / Bad Heel) → downloaded below.
3. **Mendeley multi-view fitness videos** (CC BY 4.0, good/bad form) → download manually (step 2b), squats and push-ups.
4. **REHAB24-6** (CC BY-NC 4.0, every rep marked correct/incorrect) → lunges and push-ups (step 2c).

Public data is used for **training and validation only**; `02_make_splits.py` keeps it out of the test set.

## Pilot run: no recordings of our own yet
You can train Model A now on public data alone (squats, lunges and push-ups), to test the whole pipeline and get first numbers.

1. Run the setup cells, then **2a** (Zenodo) and, if you have them, **2b** (Mendeley) and **2c** (REHAB24-6). Skip **1**.
2. In **3**, run `01b`, and run `01` if `raw/` has clips (Mendeley or REHAB24-6).
3. In **4**, run `02_make_splits.py` **without** any `--pin` (there are no own participants yet).
4. Run **5** as normal, then the **Pilot evaluation** cell instead of **6**.

Limits: Zenodo alone covers only 3 squat classes; Mendeley and REHAB24-6 add lunges and push-ups once their incorrect clips are labelled. The numbers are on validation, not test. These numbers are **not** the final results. When your recordings arrive, re-run from **3**: `01_extract_reps.py` rewrites `reps.csv`, so the pilot's split is replaced.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
D = '/content/drive/MyDrive/formcoach'
for sub in ['raw', 'public', 'review', 'processed', 'runs']:
    os.makedirs(f'{D}/{sub}', exist_ok=True)
%env D=$D

In [ ]:
!git clone -q https://github.com/marimbel/formcoach-models.git /content/formcoach-models 2>/dev/null || git -C /content/formcoach-models pull -q
%cd /content/formcoach-models
!pip -q install mediapipe
!python -m pytest -q tests

## 1. Our own recordings
Check what's in `raw/`: every own clip must start with `p` + a number (p01, p02, …).

In [ ]:
!python scripts/00_import_public_data.py inspect $D/raw --depth 1

## 2a. Zenodo squat images (automatic download, ~825 MB)
Teng (2025), *Squat Dataset*, Zenodo, doi:10.5281/zenodo.17558630, CC BY 4.0.

In [ ]:
%%bash
cd $D/public
if [ ! -d zenodo_squat ]; then
  wget -q -O squat.zip "https://zenodo.org/records/17558630/files/Dataset.zip?download=1"
  echo "cebeb32ac04dab34d4abe9686a9fc5d3  squat.zip" | md5sum -c - && unzip -q squat.zip -d zenodo_squat && rm squat.zip
fi
cd /content/formcoach-models && python scripts/00_import_public_data.py inspect $D/public/zenodo_squat --depth 3

Match the folder names printed above to our classes. Edit the `--map` values if the folders are named differently.
"Bad Back" (spinal misalignment) is mapped to `forward_lean`, the closest class we have; say so in the report.

In [ ]:
!python scripts/00_import_public_data.py images $D/public/zenodo_squat --prefix zsq \
    --map "good=squat:correct" --map "bad back=squat:forward_lean" --map "bad heel=squat:heel_lift" \
    --out $D/public/zenodo_squat_manifest.csv

## 2b. Mendeley multi-view fitness videos (manual download)
1. Open https://data.mendeley.com/datasets/kgbb3yn47p/2 → **Download All**.
2. Upload the zip to `MyDrive/formcoach/public/`, then run the next cell.

Mashetty et al. (2026), *A Multi-View Raw Video Dataset of Seven Fitness Exercises with Good/Bad Form Labels*, Mendeley Data, doi:10.17632/kgbb3yn47p.2, CC BY 4.0.

In [ ]:
%%bash
cd $D/public
Z=$(ls *.zip 2>/dev/null | grep -iv squat | head -1)
if [ -n "$Z" ] && [ ! -d mendeley ]; then unzip -q "$Z" -d mendeley; fi
cd /content/formcoach-models && python scripts/00_import_public_data.py inspect $D/public/mendeley --depth 4 | head -60

Good squat clips go straight to `raw/`. **Bad** clips have no error type: the script makes one picture per clip in `review/mendeley/` and a `review.csv`.
Open `review.csv` in Google Sheets, write the error for each row (`knee_valgus`, `forward_lean`, `insufficient_depth`, `heel_lift`, or `skip`), download it as CSV back to the same place, then run the apply cell.
If `inspect` shows a different word for squat or for good/bad, change `--keyword/--good/--bad`.

In [ ]:
!python scripts/00_import_public_data.py videos $D/public/mendeley --exercise squat --keyword squat \
    --good good --bad bad --prefix mdl --out $D/raw --review $D/review/mendeley

In [ ]:
# after labelling review.csv
!python scripts/00_import_public_data.py apply-review $D/review/mendeley/review.csv --out $D/raw

**Mendeley push-ups.** Same steps in their own review folder. Label the bad clips `hip_sag`, `elbow_flare`, `partial_rom` or `skip`.

In [ ]:
!python scripts/00_import_public_data.py videos $D/public/mendeley --exercise push_up --keyword push --good good --bad bad --prefix mdl --out $D/raw --review $D/review/mendeley_push_up
# after labelling review/mendeley_push_up/review.csv:
# !python scripts/00_import_public_data.py apply-review $D/review/mendeley_push_up/review.csv --out $D/raw

## 2c. REHAB24-6 lunges and push-ups (Zenodo, ~3.5 GB)
*REHAB24-6* (2024), Zenodo, doi:10.5281/zenodo.13305826, CC BY-NC 4.0 (non-commercial: fine for a class project; cite the SISAP 2024 paper).

Every repetition is marked correct or incorrect in `Segmentation.csv` (exercise 5 = lunge, 3 = push-up). Correct runs are cut into clips in `raw/`. Incorrect runs go to `review/rehab/<exercise>/` for you to label with the error type, like the Mendeley clips. Participants become `reh01`–`reh10`, train/val only.

If the download fails, get `videos.zip` and `Segmentation.csv` from https://zenodo.org/records/13305826, upload them to `MyDrive/formcoach/public/rehab24/`, and run the cell again.

In [ ]:
%%bash
mkdir -p $D/public/rehab24 && cd $D/public/rehab24
if [ ! -f Segmentation.csv ]; then wget -q -O Segmentation.csv "https://zenodo.org/records/13305826/files/Segmentation.csv?download=1"; fi
if [ ! -d videos ]; then
  [ -f videos.zip ] || wget -q -O videos.zip "https://zenodo.org/records/13305826/files/videos.zip?download=1"
  echo "ed183a245a1b171638e422f7a288e5a8  videos.zip" | md5sum -c - && unzip -q videos.zip -d videos && rm videos.zip
fi
cd /content/formcoach-models && python scripts/00_import_public_data.py inspect $D/public/rehab24/videos --depth 2

In [ ]:
!python scripts/00_import_public_data.py rehab $D/public/rehab24/videos --segmentation $D/public/rehab24/Segmentation.csv --map 5=lunge --map 3=push_up --prefix reh --out $D/raw --review $D/review/rehab

In [ ]:
# after labelling review/rehab/lunge/review.csv and review/rehab/push_up/review.csv:
!python scripts/00_import_public_data.py apply-review $D/review/rehab/lunge/review.csv --out $D/raw
!python scripts/00_import_public_data.py apply-review $D/review/rehab/push_up/review.csv --out $D/raw

## 3. Extract reps and crops (pose → rep segmentation → bottom-of-rep crops)
Then **open some QA plots** in `processed/qa/` and check the rep counts. Delete or re-record bad clips.

In [ ]:
!python scripts/01_extract_reps.py --videos $D/raw --out $D/processed
!python scripts/01b_extract_images.py --manifest $D/public/zenodo_squat_manifest.csv --prefix zsq --out $D/processed

In [ ]:
from IPython.display import Image, display
import glob
for f in sorted(glob.glob(f'{D}/processed/qa/*.png'))[:6]:
    display(Image(f))

## 4. Train / validation / test split
Own participants: 70/15/15 by participant. Public data: train/val only. **Pin non-members to test** (edit the ids).

In [ ]:
!python scripts/02_make_splits.py --data $D/processed --pin p09=test --pin p10=test

In [ ]:
# PILOT ONLY: public data, no own participants, so no pins and no test split
!python scripts/02_make_splits.py --data $D/processed

## 5. Train Model A: pretrained baseline vs fine-tuned

In [ ]:
!python scripts/03_train_model_a.py --data $D/processed --mode probe    --out $D/runs/probe
!python scripts/03_train_model_a.py --data $D/processed --mode finetune --out $D/runs/finetune

## 6. Evaluate on the unseen test participants

In [ ]:
!python scripts/04_eval_model_a.py --data $D/processed --runs $D/runs/probe $D/runs/finetune --export-preds $D/runs/finetune
!python scripts/04b_feature_baseline.py --data $D/processed
from IPython.display import Markdown
display(Markdown(open(f'{D}/runs/results_model_a.md').read()))
for f in ['curves.png', 'confusion.png', 'success.jpg', 'failure.jpg']:
    display(Image(f'{D}/runs/finetune/{f}'))

### Pilot evaluation (validation split; skip once you have your own test participants)

In [ ]:
!python scripts/04_eval_model_a.py --data $D/processed --runs $D/runs/probe $D/runs/finetune --eval-split val --export-preds $D/runs/finetune
!python scripts/04b_feature_baseline.py --data $D/processed --eval-split val
from IPython.display import Markdown
display(Markdown(open(f'{D}/runs/results_model_a.md').read()))
for f in ['curves.png', 'confusion.png', 'success.jpg', 'failure.jpg']:
    display(Image(f'{D}/runs/finetune/{f}'))

## 7. Export for the app + data tables for the report

In [ ]:
!python scripts/05_export_tflite.py --run $D/runs/finetune --data $D/processed --fp16
!python scripts/10_data_report.py --data $D/processed --out $D/DATA_REPORT.md
display(Markdown(open(f'{D}/DATA_REPORT.md').read()))
print('Copy', f'{D}/runs/finetune/form_classifier.tflite', '-> formcoach-app/mobile/assets/models/')